In [1]:
import pandas as pd
import pulp 
import numpy as np
import os
import pickle
from tqdm.auto import tqdm
from itertools import chain, combinations

/Users/rohitsuresh/miniconda3/envs/pytorch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:


# Load the processed optimization inputs
data_path = '../data/processed/optimization_inputs_15_farmers.pkl'
with open(data_path, 'rb') as f:
    data = pickle.load(f)

# Load the standalone values generated in Notebook 02
with open('../data/processed/standalone_values_15_farmers.pkl', 'rb') as f:
    standalone_values = pickle.load(f)

farmers = data['farmer_ids']
practices = data['practice_ids']
c = data['constants']
total_solo_value = sum(standalone_values.values())

print(f"Loaded {len(farmers)} farmers and {len(practices)} practices.")
print(f"Sum of Solo Values (from NB 02): ₹ {total_solo_value:,.2f}")

Loaded 15 farmers and 20 practices.
Sum of Solo Values (from NB 02): ₹ 0.00


In [3]:
# Cell 2: Helper Functions
def get_shared_costs(coalition_size, total_area, constants):
    """Calculates shared MRV and Transaction costs for a coalition."""
    mrv_cost = constants['MRV_FIXED'] + constants['MRV_VARIABLE'] * (total_area ** constants['MRV_DELTA'])
    t_cost = constants['T_FIXED'] + constants['T_VARIABLE'] * coalition_size
    return mrv_cost, t_cost

def powerset(iterable):
    """Generates all subsets of an iterable."""
    s = list(iterable)
    return chain.from_iterable(combinations(s, r) for r in range(len(s) + 1))

In [ ]:
# Coalition Optimization 
def optimize_coalition(coalition, data, rule='CF1'):
    """
    Solves joint optimization for a given sub-coalition.
    """
    prob = pulp.LpProblem(f"Coalition", pulp.LpMaximize)
    pairs = list(data['alpha_carbon'].keys())
    
    total_area = sum([data['farm_sizes'][f] for f in coalition])
    mrv_cost, t_cost = get_shared_costs(len(coalition), total_area, c)
    shared_cost = mrv_cost + t_cost
    
    # Variables per farmer
    x = pulp.LpVariable.dicts("x", (coalition, practices), cat='Binary')
    z = pulp.LpVariable.dicts("z", (coalition, pairs), cat='Binary')
    
    surplus = {}
    total_revenue = 0
    total_coalition_oc = 0
    total_coalition_y_loss = 0
    
    for f in coalition:
        fs = data['farm_sizes'][f]
        budget = data['farmer_budgets'][f]
        
        # Constraints per farmer
        for (j, k) in pairs:
            is_incompatible = (
                data['alpha_carbon'].get((j,k), 0) <= -1e5 or 
                data['beta_cost'].get((j,k), 0) <= -1e5 or 
                data['gamma_yield'].get((j,k), 0) <= -1e5
            )
            
            if is_incompatible:
                prob += x[f][j] + x[f][k] <= 1
                prob += z[f][(j, k)] == 0
            else:
                prob += z[f][(j, k)] <= x[f][j]
                prob += z[f][(j, k)] <= x[f][k]
                prob += z[f][(j, k)] >= x[f][j] + x[f][k] - 1
                
        f_csp = pulp.lpSum([x[f][j] * data['base_csp'][j] for j in practices]) + \
                pulp.lpSum([z[f][pair] * data['alpha_carbon'][pair] for pair in pairs])
                
        f_oc = pulp.lpSum([x[f][j] * data['base_oc'][j] for j in practices]) + \
               pulp.lpSum([z[f][pair] * data['beta_cost'][pair] for pair in pairs])
               
        f_y = pulp.lpSum([z[f][pair] * data['gamma_yield'].get(pair, 0) for pair in pairs])
        
        # 1. Individual Budget Feasibility (Must afford own ops & yield loss)
        prob += f_oc + (fs * f_y) <= budget
        
        # Financial accounting
        f_revenue = fs * f_csp * c['CCP']
        total_revenue += f_revenue
        total_coalition_oc += f_oc
        total_coalition_y_loss += (fs * f_y)
        
        # Residual surplus available to contribute to shared costs
        surplus[f] = budget - (fs * f_y) - f_oc
        
    # 2. Coalition Feasibility Constraint
    if rule == 'CF1':
        # Pooled residual must cover shared certification costs
        prob += pulp.lpSum([surplus[f] for f in coalition]) >= shared_cost
    elif rule == 'CF2':
        # Area-weighted share
        for f in coalition:
            prob += surplus[f] >= (data['farm_sizes'][f] / total_area) * shared_cost

    # 3. Objective: Maximize Grand Net Profit
    # Profit = Revenue - Yield Loss - Ops Cost - Shared Certification
    prob += total_revenue - total_coalition_y_loss - total_coalition_oc - shared_cost
    
    prob.solve(pulp.PULP_CBC_CMD(msg=False))
    
    if pulp.LpStatus[prob.status] == 'Optimal':
        return pulp.value(prob.objective)
    return -1e6 # Heavily penalize infeasible coalitions

In [5]:
# Cell 4: Verify Superadditivity (Grand Coalition Check)
print("--- Testing Grand Coalition (CF1) ---")
grand_coalition = tuple(farmers)

grand_val = optimize_coalition(grand_coalition, data, rule='CF1')
print(f"Grand Coalition Total Value: ₹ {grand_val:,.2f}")
print(f"Sum of Solo Values:          ₹ {total_solo_value:,.2f}")

surplus = grand_val - total_solo_value
print(f"Cooperative Surplus:         ₹ {surplus:,.2f}")

if surplus >= 0:
    print("✅ SUCCESS: Superadditivity holds. The coalition generates a positive surplus.")
else:
    print("❌ WARNING: The coalition performs worse than individual farmers. Stop and check constraints before running the big loop.")

--- Testing Grand Coalition (CF1) ---
Grand Coalition Total Value: ₹ 368,193.54
Sum of Solo Values:          ₹ 0.00
Cooperative Surplus:         ₹ 368,193.54
✅ SUCCESS: Superadditivity holds. The coalition generates a positive surplus.


In [ ]:
# Cell 5: Compute Characteristic Function v(T) for all sub-coalitions
# ONLY run this cell if Cell 4 showed a SUCCESS

all_subsets = list(powerset(farmers))[1:] # Exclude empty set
total_coalitions = len(all_subsets)

print(f"Total sub-coalitions to compute: {total_coalitions}")

v_dict_cf1 = {}

for subset in tqdm(all_subsets, desc="Computing v(T)"):
    coalition_tuple = tuple(subset)
    # frozenset is required so we can use it as a dictionary key later
    coalition_key = frozenset(coalition_tuple) 
    
    val = optimize_coalition(coalition_tuple, data, rule='CF1')
    
    # If a coalition is infeasible, we set its value to 0 to represent it cannot form
    v_dict_cf1[coalition_key] = max(0.0, val)

# Save the characteristic function
output_path = '../data/processed/v_dict_15_farmers_CF1.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(v_dict_cf1, f)

print(f"\nCharacteristic function successfully saved to {output_path}")

Total sub-coalitions to compute: 32767


Computing v(T):  47%|████▋     | 15411/32767 [50:37<1:07:22,  4.29it/s]